In [ ]:
from PIL import Image

In [ ]:
def crop_input_images(left, right):
    l_width, l_height = left.size
    r_width, r_height = right.size
    
    w_dif = abs(l_width - r_width)
    h_dif = abs(l_height - r_height)

    # crop the larger width, leave height alone
    if l_width > r_width:
        left_bound = w_dif / 2 - ((w_dif % 2)/2)
        right_bound = l_width - (w_dif / 2) - ((w_dif % 2)/2)
        left = left.crop((left_bound, 0, right_bound, l_height))
        l_width, l_height = left.size
        # print(f"\tcropping the left input image's width by {w_dif} pixels")
    elif r_width > l_width:
        left_bound = w_dif / 2 - ((w_dif % 2)/2)
        right_bound = r_width - (w_dif / 2) - ((w_dif % 2)/2)
        right = right.crop((left_bound, 0, right_bound, r_height))
        r_width, r_height = right.size
        # print(f"\tcropping the right input image's width by {w_dif} pixels")
    
    # crop the larger height, leave width alone
    if l_height > r_height:
        upper_bound = h_dif / 2 - ((h_dif % 2)/2)
        lower_bound = l_height - (h_dif / 2) - ((h_dif % 2)/2)
        left = left.crop((0, upper_bound, l_width, lower_bound))
        l_width, l_height = left.size
        # print(f"\tcropping the left input image's height by {h_dif} pixels")
    elif r_height > l_height:
        upper_bound = h_dif / 2 - ((h_dif % 2)/2)
        lower_bound = r_height - (h_dif / 2) - ((h_dif % 2)/2)
        right = right.crop((0, upper_bound, r_width, lower_bound))
        r_width, r_height = right.size
        # print(f"\tcropping the right input image's height by {h_dif} pixels")
    
    return(left, right)

In [ ]:
# Anaglyph function adapted from: https://github.com/miguelgrinberg/anaglyph.py/blob/master/anaglyph.py
matrices = {
    'true': [ [ 0.299, 0.587, 0.114, 0, 0, 0, 0, 0, 0 ], [ 0, 0, 0, 0, 0, 0, 0.299, 0.587, 0.114 ] ],
    'mono': [ [ 0.299, 0.587, 0.114, 0, 0, 0, 0, 0, 0 ], [ 0, 0, 0, 0.299, 0.587, 0.114, 0.299, 0.587, 0.114 ] ],
    'color': [ [ 1, 0, 0, 0, 0, 0, 0, 0, 0 ], [ 0, 0, 0, 0, 1, 0, 0, 0, 1 ] ],
    'halfcolor': [ [ 0.299, 0.587, 0.114, 0, 0, 0, 0, 0, 0 ], [ 0, 0, 0, 0, 1, 0, 0, 0, 1 ] ],
    'optimized': [ [ 0, 0.7, 0.3, 0, 0, 0, 0, 0, 0 ], [ 0, 0, 0, 0, 1, 0, 0, 0, 1 ] ],
}
def make_anaglyph(left, right, color, path):
    if left.size != right.size:
        left, right = crop_input_images(left, right)
    
    width, height = left.size
    leftMap = left.load()
    rightMap = right.load()
    m = matrices[color]

    for y in range(0, height):
        for x in range(0, width):
            r1, g1, b1 = leftMap[x, y]
            r2, g2, b2 = rightMap[x, y]
            leftMap[x, y] = (
                int(r1*m[0][0] + g1*m[0][1] + b1*m[0][2] + r2*m[1][0] + g2*m[1][1] + b2*m[1][2]),
                int(r1*m[0][3] + g1*m[0][4] + b1*m[0][5] + r2*m[1][3] + g2*m[1][4] + b2*m[1][5]),
                int(r1*m[0][6] + g1*m[0][7] + b1*m[0][8] + r2*m[1][6] + g2*m[1][7] + b2*m[1][8])
            )
    left.save(path)

In [ ]:
# mosaics

path_stem = "/Users/sabrinacurtis/Documents/M2020/anaglyphs/mosaics"
sol_seqid = [
    (1399, 4068),
    (1436, 4089),
    (1078, 3875), # left and right input mosaics are different sizes
    (1174, 3917), # left and right input mosaics are different sizes
    (1186, 3927), # left and right input mosaics are different sizes
    (1281, 3986),
    ("0831", 3681),
]

for sol, seq in sol_seqid:
    print(f"Working on sol {sol}, seq_id {seq}")
    
    mosaic_left_path = f"{path_stem}/enhanced_color_L0R_L0G_L0B_{sol}_zcam0{seq}_mosaic.png"
    mosaic_right_path = f"{path_stem}/enhanced_color_R0R_R0G_R0B_{sol}_zcam0{seq}_mosaic.png"

    # enh_left = Image.open(mosaic_left_path).convert("RGB")
    # enh_right = Image.open(mosaic_right_path).convert("RGB")
    # make_anaglyph(enh_left, enh_right, "color", f"{path_stem}/anaglyph_mosaic_{sol}_zcam0{seq}_test.png")

    try:
        enh_left = Image.open(mosaic_left_path).convert("RGB")
        enh_right = Image.open(mosaic_right_path).convert("RGB")
        make_anaglyph(enh_left, enh_right, "color", f"{path_stem}/anaglyph_mosaic_{sol}_zcam0{seq}_color.png")
    except:
        print("\t Full color anaglyph failed, skipping")
    try:
        enh_left = Image.open(mosaic_left_path).convert("RGB")
        enh_right = Image.open(mosaic_right_path).convert("RGB")
        make_anaglyph(enh_left, enh_right, "mono", f"{path_stem}/anaglyph_mosaic_{sol}_zcam0{seq}_mono.png")
    except:
        print("\t Mono color anaglyph failed, skipping")

print("Done")

In [ ]:
# concurrent stereo observations ("7L7R" in sequence name instead of "7L_7R")
# most/all are nearfield

path_stem = "/Users/sabrinacurtis/Documents/M2020/anaglyphs/concurrent/"
sol_seqid = [
    (1558, 4176, 508), # repointed
    (1558, 4175, 504), # concurrent stereo
    (1268, 3974, 318), # concurrent stereo
    (1297, 3994, 318), # concurrent stereo
    (1333, 4035, 242), # concurrent stereo
    (1357, 4046, 1264), # concurrent stereo
    (1368, 4053, 1002), # concurrent stereo
]

for sol, seq, rsm in sol_seqid:
    print(f"Working on sol {sol}, seq_id {seq}")
    sol_dir = path_stem + str(sol)
    
    enh_left_path = f"{sol_dir}/enhanced_color_L0R_L0G_L0B_SOL{sol}_zcam0{seq}_RSM{rsm}.png"
    enh_right_path = f"{sol_dir}/enhanced_color_R0R_R0G_R0B_SOL{sol}_zcam0{seq}_RSM{rsm}.png"

    try:
        enh_left = Image.open(enh_left_path).convert("RGB")
    except:
        print("\t L0 missing, falling back to L256")
        enh_left_path = f"{sol_dir}/enhanced_color_L2_L5_L6_SOL{sol}_zcam0{seq}_RSM{rsm}.png"
        enh_left = Image.open(enh_left_path).convert("RGB")
        
    enh_right = Image.open(enh_right_path).convert("RGB")
    make_anaglyph(enh_left, enh_right, "color", f"{sol_dir}/anaglyph_enhanced_color_{sol}_zcam{seq}.png")
    
    enh_left = Image.open(enh_left_path).convert("RGB")
    enh_right = Image.open(enh_right_path).convert("RGB")
    make_anaglyph(enh_left, enh_right, "mono", f"{sol_dir}/anaglyph_enhanced_mono_{sol}_zcam{seq}.png")

print("Done")

In [ ]:
path_stem = "/Users/sabrinacurtis/Documents/M2020/anaglyphs/"
sol_seqid = [
    (1536, 4165), # nearfield, variation
    (1519, 4151), # nearfield, mostly flat
    (1534, 4164), # farfield, crests and slopes
    (1515, 4147), # farfield, boulders
    (1503, 4132), # farfield, very flat
    (1456, 4101), # farfield, lots of contrast in terms of distances
    (1504, 4134), # midfield, boulders
    (1380, 4058), # midfield, mostly flat
    (1367, 4052), # midfield, mostly flat
    (1310, 4015), # nearfield
]

for sol, seq in sol_seqid:
    print(f"Working on sol {sol}, seq_id {seq}")
    sol_dir = path_stem + str(sol)

    # nat_left_path = f"{sol_dir}/natural color L0R L0G L0B.jpg"
    # nat_right_path = f"{sol_dir}/natural color R0R R0G R0B.jpg"
    enh_left_path = f"{sol_dir}/enhanced color L0R L0G L0B.jpg"
    enh_right_path = f"{sol_dir}/enhanced color R0R R0G R0B.jpg"
    
    enh_left = Image.open(enh_left_path) # add .convert("RGB") if the input image is .png
    enh_right = Image.open(enh_right_path)
    make_anaglyph(enh_left, enh_right, "color", f"{sol_dir}/anaglyph_enhanced_color_{sol}_zcam{seq}.png")
    
    enh_left = Image.open(enh_left_path)
    enh_right = Image.open(enh_right_path)
    make_anaglyph(enh_left, enh_right, "mono", f"{sol_dir}/anaglyph_enhanced_mono_{sol}_zcam{seq}.png")

    # nat_left = Image.open(nat_left_path)
    # nat_right = Image.open(nat_right_path)
    # make_anaglyph(nat_left, nat_right, "color", f"{sol_dir}/anaglyph_natural_color_{sol}_zcam{seq}.png")
    
    # enh_left = Image.open(enh_left_path)
    # enh_right = Image.open(enh_right_path)
    # make_anaglyph(enh_left, enh_right, "optimized", f"{sol_dir}/anaglyph_enhanced_partial_{sol}_zcam{seq}.png")

print("Done")